# 02 — Preprocessing : CIC-IDS2017

## 🎯 Objectif de ce notebook

Transformer le dataset brut (2,83M lignes, sales, déséquilibrées) en un dataset propre et exploitable par nos futurs modèles ML/DL.

## 📋 Plan

1. **Chargement** des 8 CSV bruts
2. **Nettoyage cosmétique** : noms de colonnes, labels corrompus
3. **Suppression des aberrations** : doublons, NaN, infinis, valeurs négatives impossibles
4. **Regroupement des classes** : 15 → 7 classes
5. **Encodage** : conversion des labels texte en nombres
6. **Sauvegarde** au format Parquet (10x plus rapide à recharger que CSV)

## ⚠️ Note pédagogique

Chaque étape contient :
- Une explication **pourquoi** on fait ça
- Le code avec des commentaires
- Un check **après** pour vérifier qu'on n'a pas fait n'importe quoi

Tu peux exécuter cellule par cellule pour bien comprendre, ou tout d'un coup avec **Run All**.

## 1. Setup et chargement

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# Affichage : large mais lisible
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

# Chemins
DATA_RAW = Path('../data/raw/cic-ids-2017')
DATA_PROCESSED = Path('../data/processed')
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print('Dossier brut :', DATA_RAW.resolve())
print('Dossier traité :', DATA_PROCESSED.resolve())

Dossier brut : C:\Users\natha\Documents\PFE\ids-xai-pfe\data\raw\cic-ids-2017
Dossier traité : C:\Users\natha\Documents\PFE\ids-xai-pfe\data\processed


### 📚 Pourquoi on recharge les données ?

Tu te demandes peut-être : *« Mais on a déjà chargé les données dans le notebook 01, pourquoi recommencer ? »*

**Réponse :** chaque notebook doit être **autonome**. Imagine que ton encadrant ou un futur recruteur veuille rejouer juste le preprocessing : il ouvre `02_preprocessing.ipynb`, il fait Run All, et ça marche. Si le notebook dépendait d'un état créé ailleurs, ce serait impossible. C'est une bonne pratique appelée **reproductibilité**.

In [2]:
# Chargement des 8 CSV (prend 1-3 min)
csv_files = sorted(DATA_RAW.glob('*.csv'))

dfs = []
for f in csv_files:
    df_tmp = pd.read_csv(f, low_memory=False)
    print(f'{f.name:60s} -> {df_tmp.shape}')
    dfs.append(df_tmp)

df = pd.concat(dfs, ignore_index=True)
print(f'\n✅ Dataset complet chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv             -> (225745, 79)
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv         -> (286467, 79)
Friday-WorkingHours-Morning.pcap_ISCX.csv                    -> (191033, 79)
Monday-WorkingHours.pcap_ISCX.csv                            -> (529918, 79)
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv  -> (288602, 79)
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv       -> (170366, 79)
Tuesday-WorkingHours.pcap_ISCX.csv                           -> (445909, 79)
Wednesday-workingHours.pcap_ISCX.csv                         -> (692703, 79)

✅ Dataset complet chargé : 2,830,743 lignes × 79 colonnes


## 2. Nettoyage cosmétique

Avant tout calcul, on s'assure que les noms de colonnes et de classes sont propres. C'est la base : si on cherche `'Label'` mais que la colonne s'appelle `' Label '` (avec des espaces), tout plante.

In [3]:
# 2.1 — Nettoyer les noms de colonnes (espaces parasites)
n_before = len(df.columns)
df.columns = df.columns.str.strip()
print(f'Colonnes nettoyées : {n_before} colonnes')

# 2.2 — Nettoyer les valeurs des labels (caractère � corrompu sur les Web Attack)
# On remplace le caractère de remplacement Unicode par un tiret propre
df['Label'] = df['Label'].str.replace('\ufffd', '-', regex=False)
df['Label'] = df['Label'].str.strip()

# Vérifie ce qu'on a maintenant
print('\n✅ Classes après nettoyage :')
print(df['Label'].value_counts())

Colonnes nettoyées : 79 colonnes

✅ Classes après nettoyage :
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack - Brute Force         1507
Web Attack - XSS                  652
Infiltration                       36
Web Attack - Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64


Tu dois maintenant voir les classes `Web Attack - Brute Force`, `Web Attack - XSS`, `Web Attack - Sql Injection` avec un vrai tiret au lieu du caractère bizarre. ✅

## 3. Suppression des aberrations

Maintenant on vire ce qui est cassé.

### 📚 Pourquoi cet ordre précis ?

On va supprimer :
1. La **colonne dupliquée** (`Fwd Header Length.1`) → on commence par là car c'est une feature inutile
2. Les lignes avec des **valeurs infinies**
3. Les lignes avec des **NaN**
4. Les lignes avec des **durées négatives** (impossibles physiquement)
5. Les **doublons**

**Pourquoi les doublons en dernier ?** Parce que si on les supprime en premier, certains des NaN/infinis pourraient se retrouver "seuls" sans leurs jumeaux, et on les manquerait. Toujours nettoyer **puis** dédupliquer.

In [4]:
# 3.1 — Suppression de la colonne dupliquée
# 'Fwd Header Length.1' est strictement identique à 'Fwd Header Length'
# (artefact du dataset original)
n_before = df.shape[1]
df = df.drop(columns=['Fwd Header Length.1'], errors='ignore')
print(f'Colonnes : {n_before} -> {df.shape[1]}')

Colonnes : 79 -> 78


In [5]:
# 3.2 — Remplacer les infinis par NaN (étape technique)
# Pandas gère mieux les NaN que les infinis : on convertit pour traiter d'un coup
n_before = len(df)
df = df.replace([np.inf, -np.inf], np.nan)
print(f'Conversion infinis → NaN faite')
print(f'Total NaN après conversion : {df.isna().sum().sum():,}')

Conversion infinis → NaN faite
Total NaN après conversion : 5,734


In [6]:
# 3.3 — Suppression des lignes contenant des NaN
n_before = len(df)
df = df.dropna()
n_dropped = n_before - len(df)
print(f'Lignes supprimées (NaN/inf) : {n_dropped:,} ({100*n_dropped/n_before:.2f}%)')
print(f'Lignes restantes : {len(df):,}')

Lignes supprimées (NaN/inf) : 2,867 (0.10%)
Lignes restantes : 2,827,876


In [7]:
# 3.4 — Suppression des valeurs négatives aberrantes sur les durées
# Une durée ne peut JAMAIS être négative. Si c'est le cas, c'est un bug de capture.
duration_cols = [c for c in df.columns if 'Duration' in c or 'IAT' in c]
print(f'Colonnes de durée à vérifier : {duration_cols}')

n_before = len(df)
# On garde uniquement les lignes où TOUTES les durées sont >= 0
mask_valid = (df[duration_cols] >= 0).all(axis=1)
df = df[mask_valid].copy()
n_dropped = n_before - len(df)
print(f'Lignes supprimées (durées négatives) : {n_dropped:,}')
print(f'Lignes restantes : {len(df):,}')

Colonnes de durée à vérifier : ['Flow Duration', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min']
Lignes supprimées (durées négatives) : 2,890
Lignes restantes : 2,824,986


In [8]:
# 3.5 — Suppression des doublons (le gros morceau : ~9% des données)
n_before = len(df)
df = df.drop_duplicates()
n_dropped = n_before - len(df)
print(f'Doublons supprimés : {n_dropped:,} ({100*n_dropped/n_before:.2f}%)')
print(f'Lignes restantes : {len(df):,}')

Doublons supprimés : 307,068 (10.87%)
Lignes restantes : 2,517,918


### 📊 Bilan du nettoyage

À ce stade, on devrait être passé de **2 830 743** lignes à environ **2,5 - 2,56 millions** de lignes propres.

In [9]:
print(f'📊 Dataset après nettoyage : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')
print(f'\nDistribution des classes :')
print(df['Label'].value_counts())

📊 Dataset après nettoyage : 2,517,918 lignes × 78 colonnes

Distribution des classes :
Label
BENIGN                        2092370
DoS Hulk                       172688
DDoS                           127995
PortScan                        90694
DoS GoldenEye                   10281
FTP-Patator                      5927
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3217
Bot                              1948
Web Attack - Brute Force         1470
Web Attack - XSS                  652
Infiltration                       35
Web Attack - Sql Injection         21
Heartbleed                          7
Name: count, dtype: int64


## 4. Regroupement des classes

### 📚 Pourquoi ?

On l'a vu en EDA : certaines classes ont **moins de 50 exemples** (Heartbleed: 11, Infiltration: 36, Web Attack SQL: 21). Impossible d'apprendre quoi que ce soit avec si peu de données.

**Notre stratégie :** regrouper les classes sémantiquement proches, et exclure celles qui ne peuvent pas être apprises.

**Important pour ton rapport :** ce choix méthodologique sera **justifié explicitement** dans la section méthodologie. C'est une bonne pratique scientifique.

In [10]:
# Mapping des classes : ancien nom -> nouveau nom
class_mapping = {
    'BENIGN': 'BENIGN',
    
    # Tous les DoS regroupés (variantes d'une même attaque)
    'DoS Hulk': 'DoS',
    'DoS GoldenEye': 'DoS',
    'DoS slowloris': 'DoS',
    'DoS Slowhttptest': 'DoS',
    
    # DDoS reste séparé (attaque distribuée, mécanisme différent)
    'DDoS': 'DDoS',
    
    # PortScan reste seul
    'PortScan': 'PortScan',
    
    # Brute Force regroupées
    'FTP-Patator': 'Brute Force',
    'SSH-Patator': 'Brute Force',
    
    # Web Attacks regroupées
    'Web Attack - Brute Force': 'Web Attack',
    'Web Attack - XSS': 'Web Attack',
    'Web Attack - Sql Injection': 'Web Attack',
    
    # Bot reste seul
    'Bot': 'Bot',
    
    # Classes EXCLUES (trop peu d'exemples)
    # Heartbleed : 11 exemples
    # Infiltration : 36 exemples
}

# Application du mapping
df['Label_grouped'] = df['Label'].map(class_mapping)

# Suppression des lignes dont la classe n'est pas dans le mapping (Heartbleed, Infiltration)
n_before = len(df)
df = df.dropna(subset=['Label_grouped']).copy()
n_dropped = n_before - len(df)
print(f'Lignes exclues (Heartbleed + Infiltration) : {n_dropped}')

# On remplace l'ancienne colonne par la nouvelle
df['Label'] = df['Label_grouped']
df = df.drop(columns=['Label_grouped'])

print(f'\n✅ Distribution finale après regroupement :')
print(df['Label'].value_counts())

Lignes exclues (Heartbleed + Infiltration) : 42

✅ Distribution finale après regroupement :
Label
BENIGN         2092370
DoS             193582
DDoS            127995
PortScan         90694
Brute Force       9144
Web Attack        2143
Bot               1948
Name: count, dtype: int64


## 5. Encodage des labels

### 📚 Pourquoi ?

Les modèles ML ne comprennent **que les nombres**, pas les textes. On doit donc convertir nos 7 classes textuelles en entiers :
- `BENIGN` → 0
- `DoS` → 1
- `DDoS` → 2
- ...

On utilise `LabelEncoder` de scikit-learn et on **sauvegarde le mapping** pour pouvoir retrouver le nom textuel plus tard (utile pour la visualisation et le rapport).

In [11]:
from sklearn.preprocessing import LabelEncoder
import json

le = LabelEncoder()
df['Label_encoded'] = le.fit_transform(df['Label'])

# Sauvegarder le mapping pour usage ultérieur
label_mapping = {int(i): label for i, label in enumerate(le.classes_)}
print('Mapping classes → entiers :')
for k, v in label_mapping.items():
    n = (df['Label_encoded'] == k).sum()
    print(f'  {k} : {v:15s}  ({n:>10,} exemples)')

# On sauvegarde le mapping dans un fichier JSON pour le retrouver facilement
with open(DATA_PROCESSED / 'label_mapping.json', 'w') as f:
    json.dump(label_mapping, f, indent=2)
print(f'\n✅ Mapping sauvegardé dans {DATA_PROCESSED}/label_mapping.json')

Mapping classes → entiers :
  0 : BENIGN           ( 2,092,370 exemples)
  1 : Bot              (     1,948 exemples)
  2 : Brute Force      (     9,144 exemples)
  3 : DDoS             (   127,995 exemples)
  4 : DoS              (   193,582 exemples)
  5 : PortScan         (    90,694 exemples)
  6 : Web Attack       (     2,143 exemples)

✅ Mapping sauvegardé dans ..\data\processed/label_mapping.json


## 6. Sauvegarde finale

### 📚 Pourquoi Parquet et pas CSV ?

Parquet est un format binaire optimisé pour les gros datasets :
- **10x plus rapide** à charger qu'un CSV
- **5x plus petit** sur disque (compression)
- **Conserve les types** (pas de devinette "est-ce un int ou un float ?")

C'est devenu le standard de fait pour le data science en production.

In [12]:
# On retire la colonne 'source_file' qui ne sert plus à rien pour le modèle
if 'source_file' in df.columns:
    df = df.drop(columns=['source_file'])

# Sauvegarde
output_path = DATA_PROCESSED / 'cic-ids-2017_clean.parquet'
df.to_parquet(output_path, index=False, engine='pyarrow', compression='snappy')

# Vérification taille
size_mb = output_path.stat().st_size / 1024**2
print(f'✅ Dataset propre sauvegardé : {output_path}')
print(f'   Taille sur disque : {size_mb:.1f} MB')
print(f'   Shape : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')

✅ Dataset propre sauvegardé : ..\data\processed\cic-ids-2017_clean.parquet
   Taille sur disque : 301.9 MB
   Shape : 2,517,876 lignes × 79 colonnes


## 7. Vérification finale

On recharge le fichier sauvegardé pour vérifier que tout est OK. C'est un réflexe à avoir : ne JAMAIS faire confiance aveuglément à une sauvegarde.

In [13]:
# Reload pour vérifier
df_check = pd.read_parquet(output_path)

print(f'📊 Dataset rechargé : {df_check.shape}')
print(f'\n✅ Distribution des classes :')
print(df_check['Label'].value_counts())
print(f'\n✅ Plus aucun NaN : {df_check.isna().sum().sum() == 0}')
print(f'✅ Plus aucun infini : {np.isinf(df_check.select_dtypes(include=[np.number])).sum().sum() == 0}')
print(f'✅ Aperçu :')
df_check.head(3)

📊 Dataset rechargé : (2517876, 79)

✅ Distribution des classes :
Label
BENIGN         2092370
DoS             193582
DDoS            127995
PortScan         90694
Brute Force       9144
Web Attack        2143
Bot               1948
Name: count, dtype: int64

✅ Plus aucun NaN : True
✅ Plus aucun infini : True
✅ Aperçu :


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,Bwd Packet Length Mean,Bwd Packet Length Std,Flow Bytes/s,Flow Packets/s,Flow IAT Mean,Flow IAT Std,Flow IAT Max,Flow IAT Min,Fwd IAT Total,Fwd IAT Mean,Fwd IAT Std,Fwd IAT Max,Fwd IAT Min,Bwd IAT Total,Bwd IAT Mean,Bwd IAT Std,Bwd IAT Max,Bwd IAT Min,Fwd PSH Flags,Bwd PSH Flags,Fwd URG Flags,Bwd URG Flags,Fwd Header Length,Bwd Header Length,Fwd Packets/s,Bwd Packets/s,Min Packet Length,Max Packet Length,Packet Length Mean,Packet Length Std,Packet Length Variance,FIN Flag Count,SYN Flag Count,RST Flag Count,PSH Flag Count,ACK Flag Count,URG Flag Count,CWE Flag Count,ECE Flag Count,Down/Up Ratio,Average Packet Size,Avg Fwd Segment Size,Avg Bwd Segment Size,Fwd Avg Bytes/Bulk,Fwd Avg Packets/Bulk,Fwd Avg Bulk Rate,Bwd Avg Bytes/Bulk,Bwd Avg Packets/Bulk,Bwd Avg Bulk Rate,Subflow Fwd Packets,Subflow Fwd Bytes,Subflow Bwd Packets,Subflow Bwd Bytes,Init_Win_bytes_forward,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label,Label_encoded
0,54865,3,2,0,12,0,6,6,6.0,0.0,0,0,0.0,0.0,4.000000e+06,666666.66670,3.0,0.0,3,3,3,3.0,0.0,3,3,0,0.0,0.0,0,0,0,0,0,0,40,0,666666.666700,0.000000,6,6,6.0,0.0,0.0,0,0,0,0,1,0,0,0,0,9.0,6.0,0.0,0,0,0,0,0,0,2,12,0,0,33,-1,1,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
1,55054,109,1,1,6,6,6,6,6.0,0.0,6,6,6.0,0.0,1.100917e+05,18348.62385,109.0,0.0,109,109,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,9174.311927,9174.311927,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,1,9.0,6.0,6.0,0,0,0,0,0,0,1,6,1,6,29,256,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0
2,55055,52,1,1,6,6,6,6,6.0,0.0,6,6,6.0,0.0,2.307692e+05,38461.53846,52.0,0.0,52,52,0,0.0,0.0,0,0,0,0.0,0.0,0,0,0,0,0,0,20,20,19230.769230,19230.769230,6,6,6.0,0.0,0.0,0,0,0,0,1,1,0,0,1,9.0,6.0,6.0,0,0,0,0,0,0,1,6,1,6,29,256,0,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN,0


## 🎉 Récapitulatif

**Avant :** 2 830 743 lignes, sales, 15 classes très déséquilibrées

**Après :** ~2,5M lignes propres, 7 classes utilisables, sauvegardées au format optimal

**Fichiers produits :**
- `data/processed/cic-ids-2017_clean.parquet` : dataset propre
- `data/processed/label_mapping.json` : correspondance classes ↔ entiers

### Prochaine étape (notebook 03)

On va commencer à **modéliser** : entraîner un premier Random Forest comme baseline, puis un XGBoost optimisé. C'est là que les choses sérieuses commencent.

In [2]:
import json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
nb_path = ROOT / "notebooks" / "02_preprocessing.ipynb"

with open(nb_path, encoding="utf-8") as f:
    nb = json.load(f)

out = []
for i, cell in enumerate(nb["cells"]):
    if cell["cell_type"] == "code":
        src = "".join(cell["source"])
        if src.strip():
            out.append(f"# ---------- Cellule {i} ----------\n{src}\n")

# Écrit tout dans un fichier à la racine du projet
dest = ROOT / "02_preprocessing_code.txt"
dest.write_text("\n".join(out), encoding="utf-8")
print(f"✅ Code complet écrit dans : {dest}")
print(f"   ({len(out)} cellules de code)")

✅ Code complet écrit dans : c:\Users\natha\Documents\PFE\ids-xai-pfe\02_preprocessing_code.txt
   (14 cellules de code)
